In [116]:
from sympy import *
from sympy.parsing.mathematica import parse_mathematica
import numpy as np
import states_and_witnesses as sw
import operations as op

In [117]:
# Define kets in vector form 
H = op.ket([1,0])
V = op.ket([0,1])
R = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (1j)])
L = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (-1j)])
D = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (1)])
A = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (-1)])

In [118]:
p = Symbol("p", positive=True)
q = Symbol("q", positive=True)

In [119]:
def phi1(alpha, alpha_perp, chi, phase=0):
    """
    phi1 is the target state we are creating with probability p.
    """
    return cos(chi/2) * np.kron(H, alpha) + exp(1j * phase) * sin(chi/2) * np.kron(V, alpha_perp)

def phi2(alpha):
    """
    phi2 is the state |H>|alpha> that we create with probability (1-p)cos^2(chi/2).
    """
    return np.kron(H, alpha)

def phi3(alpha_perp):
    """
    phi3 is the state |V>|alpha_perp> that we create with probability (1-p)sin^2(chi/2).
    """
    return np.kron(V, alpha_perp)

In [120]:
def rho(p, alpha, alpha_perp, chi, phase=0):
    """
    Constructs the density matrix based on the probabilities of each state created.
    """
    target_rho = p * (phi1(alpha, alpha_perp, chi, phase) * op.adjoint(phi1(alpha, alpha_perp, chi, phase)))
    phi2_rho = (1-p)*(cos(chi/2))**2 * np.outer(np.kron(H, alpha), op.adjoint(np.kron(H, alpha)))
    phi3_rho = (1-p)*(sin(chi/2))**2 * np.outer(np.kron(V, alpha_perp), op.adjoint(np.kron(V, alpha_perp)))
    return target_rho + phi2_rho + phi3_rho

In [121]:
rho1 = rho(p, H, V, pi/2)
print(rho1)
rho1_sq = rho1 @ rho1
print("Trace of rho squared:", np.trace(rho1_sq))

[[0.500000000000000 0 0 0.5*p]
 [0 0 0 0]
 [0 0 0 0]
 [0.5*p 0 0 0.500000000000000]]
Trace of rho squared: 0.5*p**2 + 0.5


In [122]:
rho2 = rho(p, D, A, pi/2)
print(rho2)
rho2_sq = rho2 @ rho2
print(rho2_sq)
print("Trace of rho squared:", np.trace(rho2_sq))

[[0.250000000000000 0.250000000000000 0.25*p -0.25*p]
 [0.250000000000000 0.250000000000000 0.25*p -0.25*p]
 [0.25*p 0.25*p 0.250000000000000 -0.250000000000000]
 [-0.25*p -0.25*p -0.250000000000000 0.250000000000000]]
[[0.125*p**2 + 0.125 0.125*p**2 + 0.125 0.25*p -0.25*p]
 [0.125*p**2 + 0.125 0.125*p**2 + 0.125 0.25*p -0.25*p]
 [0.25*p 0.25*p 0.125*p**2 + 0.125 -0.125*p**2 - 0.125]
 [-0.25*p -0.25*p -0.125*p**2 - 0.125 0.125*p**2 + 0.125]]
Trace of rho squared: 0.5*p**2 + 0.5


In [123]:
rho3 = rho(p, R, L, 0.5*pi)
rho3_sq = rho3 @ rho3
print("Trace of rho squared:", np.trace(rho3_sq))

Trace of rho squared: 0.5*p**2 + 4*(-0.25*I*p - 0.5*I*(1/2 - p/2))*(0.25*I*p + 0.5*I*(1/2 - p/2)) + 0.25


In [124]:
rho4 = rho(p, H, V, 0.25*pi)
rho4_sq = rho4 @ rho4
print(
np.trace(rho4_sq))

2.0*p**2*(1/2 - sqrt(2)/4)*(sqrt(2)/4 + 1/2) + (p*(0.5 - 0.25*sqrt(2)) + 1.0*(1/2 - sqrt(2)/4)*(1 - p))**2 + (p*(0.25*sqrt(2) + 0.5) + 1.0*(1 - p)*(sqrt(2)/4 + 1/2))**2


In [125]:
equation = Eq(np.trace(rho4_sq), q)
print(solveset(equation, p))

{-2.0*sqrt(1.0*q - 0.75), 2.0*sqrt(1.0*q - 0.75)}


In [126]:
rho5 = rho(p, H, V, 0.001)
rho5_sq = rho5 @ rho5
print(np.trace(rho5_sq))
equation = Eq(np.trace(rho5_sq), q)
print(solveset(equation, p))

4.99999833333356e-7*p**2 + 0.999999500000167
{-1414.21379807538*sqrt(1.0*q - 0.999999500000167), 1414.21379807538*sqrt(1.0*q - 0.999999500000167)}


In [129]:
chis = np.linspace(0.001, np.pi/2, 6)
for chi in chis:
    this_rho = rho(p, H, V, chi)
    tr_rho_sq = np.trace(this_rho @ this_rho)
    print(f"\nchi: {np.rad2deg(chi)}, trace(rho^2): {tr_rho_sq}")
    equation = Eq(tr_rho_sq, q)
    print(f"chi: {np.rad2deg(chi)}, p: {solveset(equation, p)}")


chi: 0.057295779513082325, trace(rho^2): 4.99999833333356e-7*p**2 + 0.999999500000167
chi: 0.057295779513082325, p: {-1414.21379807538*sqrt(1.0*q - 0.999999500000167), 1414.21379807538*sqrt(1.0*q - 0.999999500000167)}

chi: 18.045836623610466, trace(rho^2): 0.0479811242922478*p**2 + 0.952018875707752
chi: 18.045836623610466, p: {-4.56525236298862*sqrt(1.0*q - 0.952018875707752), 4.56525236298862*sqrt(1.0*q - 0.952018875707752)}

chi: 36.03437746770785, trace(rho^2): 0.173031123915728*p**2 + 0.826968876084272
chi: 36.03437746770785, p: {-2.40401894394615*sqrt(1.0*q - 0.826968876084272), 2.40401894394615*sqrt(1.0*q - 0.826968876084272)}

chi: 54.022918311805235, trace(rho^2): 0.327444435155348*p**2 + 0.672555564844652
chi: 54.022918311805235, p: {-1.74755636800476*sqrt(1.0*q - 0.672555564844652), 1.74755636800476*sqrt(1.0*q - 0.672555564844652)}

chi: 72.01145915590261, trace(rho^2): 0.452313010937059*p**2 + 0.547686989062941
chi: 72.01145915590261, p: {-1.48689554324964*sqrt(1.0*q - 0.